<a href="https://colab.research.google.com/github/kamejoko80/optispeech/blob/henry_rk3588/notebooks/henry_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Python 3.11

In [ ]:
# 1. Install Python 3.11 and Dev tools globally
!apt-get update
!apt-get install python3.11 python3.11-dev python3.11-distutils -y

# 2. Register both versions in the 'update-alternatives' system
# We give Python 3.11 a higher priority (2) than Python 3.12 (1)
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1

# 3. Fix Pip (changing Python versions often breaks the global pip link)
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 4. Verify the change
!python3 --version


# Environment Settings

In [ ]:
%cd /content

!pip install -U pip
!git clone -b henry_rk3588 https://github.com/kamejoko80/optispeech.git

# Install OptiSpeech
%cd /content/optispeech
!pip install -e .

# Install some extra dependencies
!pip install transformers -U
!pip install onnxruntime soundfile numpy
!pip install torch torchvision torchaudio
!python3 -c "import torch; print(torch.__version__)"

%cd /content/optispeech/rknn_rk3588

# Create models and out folders
import os
folders = ['models', 'out']
for folder in folders:
    if os.path.exists(folder):
        print(f"✅ Folder '{folder}' already exists.")
    else:
        os.makedirs(folder)
        print(f"📁 Folder '{folder}' was created.")

        # Download the prestrained checkpoint
        from huggingface_hub import hf_hub_download
        repo_id = "henrydang80/optispeech"
        filename = "checkpoints/lightspeech/en-us/mike-checkpoint_epoch-729_step-305000.ckpt"
        path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="models", local_dir_use_symlinks=False)
        print("Saved to:", path)




# Model Inference Example

In [ ]:
%cd /content/optispeech/rknn_rk3588
!python3 test_pytorch.py

from IPython.display import Audio
Audio('output.wav')

# Prepare Dataset For The Model Training

In [ ]:
%cd /content/optispeech

import shutil
import os

def copy_drive_file(source_path, dest_folder="/content/"):
    """
    Copies a file from Google Drive to a destination in Colab.

    Args:
        source_path (str): Full path to the file in Google Drive.
        dest_folder (str): Destination directory in Colab.
    """
    # Ensure the destination folder exists
    os.makedirs(dest_folder, exist_ok=True)

    try:
        # Perform the copy
        shutil.copy(source_path, dest_folder)
        filename = os.path.basename(source_path)
        print(f"✅ Success: '{filename}' copied to '{dest_folder}'")
        return True # Return True if successful

    except FileNotFoundError:
        print(f"❌ Error: Source file not found at: {source_path}")
    except PermissionError:
        print("❌ Error: Permission denied. Make sure your Google Drive is mounted.")
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")

    return False # Return False if it failed


# Create datasets folder
import os

folders = ['datasets']

for folder in folders:
    if os.path.exists(folder):
        print(f"✅ Folder '{folder}' already exists.")
    else:
        os.makedirs(folder)
        print(f"📁 Folder '{folder}' was created.")

        # Dowload the datasets
        %cd datasets

        # Download from network
        # !wget -O LJSpeech-1.1.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2

        # Mount the Gdrive
        from google.colab import drive
        drive.mount('/content/drive')

        # Copy from Gdrive
        copy_drive_file(
          source_path='/content/drive/My Drive/Local_AI/LJSpeech-1.1.tar.bz2',
          dest_folder='/content/optispeech/datasets'
        )

        # Untar the package
        !tar -xjf LJSpeech-1.1.tar.bz2

        # Resample the wav files to 24K
        %cd /content/optispeech
        !python3 scripts/resample_ljspeech_to_24k.py

        # Covert ljspeech datasets to Hydra fortmat
        !rm -rf data/hfc_female-en_us/input
        !python3 scripts/convert_ljspeech_to_optispeech_input.py \
                 --ljspeech_dir datasets/LJSpeech-1.1 \
                 --out_input_dir data/hfc_female-en_us/input \
                 --val_size 500



# Execute Datasets Preparing

In [ ]:
%cd /content/optispeech

# Run the preprocess_dataset
!rm -rf data/hfc_female-en_us/output
!python3 -m optispeech.tools.preprocess_dataset \
         --format ljspeech \
         -w 8 -b 1 \
         hfc_female-en_us \
         data/hfc_female-en_us/input \
         data/hfc_female-en_us/output


# Start Training

In [ ]:
%cd /content/optispeech

!export CUDA_VISIBLE_DEVICES=0 \
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:32,garbage_collection_threshold:0.8 \
python3 -m optispeech.train experiment=hfc_female-en_us \
  run_name=opti_hfc_female_gpu \
  data.train_filelist_path=data/hfc_female-en_us/output/train.safe.txt \
  data.valid_filelist_path=data/hfc_female-en_us/output/val.safe.txt \
  data.batch_size=1 data.num_workers=2 data.pin_memory=true \
  model.train_args.gradient_accumulate_batches=64 \
  trainer.accelerator=gpu trainer.devices=1 trainer.precision=16-mixed \
  +trainer.num_sanity_val_steps=0 +trainer.limit_val_batches=0.0 \
  +trainer.max_steps=300000 \
  model.generator.segment_size=16 \
  model.train_args.evaluate_utmos=false \
  model.train_args.evaluate_pesq=false \
  model.train_args.evaluate_periodicity=false \
  callbacks.model_checkpoint.save_last=true